# Ingestão de Dados - Camada 01_RAW

Este notebook tem como objetivo realizar a ingestão dos dados brutos no ambiente Databricks, salvando-os em formato Delta na camada `01_raw`.

##  Fontes de dados
- vendas_2023_2024.csv
- produtos_raw.csv
- clientes_crm.json
- custos_importacao.json

##  Objetivo
- Ler arquivos armazenados no Volume (`raw_files`)
- Inferir schema automaticamente
- Persistir como tabelas Delta no catálogo `lh_nautical`
- Garantir reprodutibilidade e rastreabilidade dos dados

In [0]:
# Importação das bibliotecas necessárias
from pyspark.sql import SparkSession


spark = SparkSession.builder.getOrCreate()

In [0]:
# Caminho base do volume onde os arquivos foram carregados
BASE_PATH = "/Volumes/lh_nautical/01_raw/raw_files/"

# Dicionário com os caminhos dos arquivos
paths = {
    "vendas": BASE_PATH + "vendas_2023_2024.csv",
    "produtos": BASE_PATH + "produtos_raw.csv",
    "clientes": BASE_PATH + "clientes_crm.json",
    "importacao": BASE_PATH + "custos_importacao.json"
}

# Exibir paths para conferência
paths

In [0]:
# Leitura do arquivo CSV de vendas
df_vendas = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(paths["vendas"])
)

# Escrita como tabela Delta na camada RAW
df_vendas.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.01_raw.vendas_raw")

# Validação rápida
print(f"Total de registros em vendas_raw: {df_vendas.count()}")
df_vendas.show(5)

In [0]:
# Leitura do arquivo CSV de produtos
df_produtos = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(paths["produtos"])
)

# Escrita como tabela Delta
df_produtos.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.01_raw.produtos_raw")

# Validação
print(f"Total de registros em produtos_raw: {df_produtos.count()}")
df_produtos.show(5)

In [0]:
# Leitura do JSON de clientes (multiline = True por ser JSON estruturado)
df_clientes = (
    spark.read
    .option("multiline", True)
    .json(paths["clientes"])
)

# Escrita como tabela Delta
df_clientes.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.01_raw.clientes_raw")

# Validação
print(f"Total de registros em clientes_raw: {df_clientes.count()}")
df_clientes.show(5)

In [0]:
# Leitura do JSON de custos de importação
df_importacao = (
    spark.read
    .option("multiline", True)
    .json(paths["importacao"])
)

# Escrita como tabela Delta
df_importacao.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.01_raw.custos_importacao_raw")

# Validação
print(f"Total de registros em custos_importacao_raw: {df_importacao.count()}")
df_importacao.show(5)

In [0]:
# Verificação rápida das tabelas criadas
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.01_raw.vendas_raw").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.01_raw.produtos_raw").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.01_raw.clientes_raw").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.01_raw.custos_importacao_raw").show()

In [0]:
df_importacao.selectExpr("explode(historic_data)").count()